In [ ]:
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
load_dotenv()

db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')
db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')

engine = create_engine(
  f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}',
  connect_args={'client_encoding': 'utf-8'}
  )

In [ ]:
expenses = pd.read_sql('SELECT * FROM finance.v_real_expenses', engine)
inflation = pd.read_sql('SELECT * FROM finance.inflation', engine)

In [ ]:
expenses.head(10)

In [ ]:
inflation.head(10)

In [ ]:
expenses.dtypes 

In [ ]:
expenses['transaction_date'] = pd.to_datetime(expenses['transaction_date'])
expenses.dtypes

In [ ]:
expenses['mcc'].isna().sum()
expenses.info()

In [ ]:
expenses['category'].isna().sum()

In [ ]:
expenses['merchant'].isna().sum()

In [ ]:
inflation.dtypes
inflation['period'] = pd.to_datetime(inflation['period'])
inflation.dtypes

In [ ]:
expenses_copy = expenses.copy()
expenses_copy['transaction_date'] = expenses_copy['transaction_date'].dt.to_period('M')
expenses_by_month = (
  expenses_copy
  .groupby('transaction_date')['amount']
  .sum()
  .reset_index()
  )
expenses_by_month

In [ ]:
inflation_copy = inflation.copy()
inflation_copy['period'] = inflation_copy['period'].dt.to_period('M')
inflation_copy

In [ ]:
merge_df = expenses_by_month.merge(inflation_copy, left_on='transaction_date', right_on='period', how='inner')
merge_df

In [ ]:
corr_analysis = merge_df[['amount', 'inflation_rate', 'key_rate']].corr()
corr_analysis

### Вывод: корреляция расходов с инфляцией и ключевой ставкой
   

Коэффициент корреляции между помесячными расходами и инфляцией составил **0.34**, 
между расходами и ключевой ставкой — **0.24**. Данный результат подтверждает сильную зависимость между расходами и макроэкономическими показателями.

In [ ]:
expenses_index_df = expenses_by_month.copy()
expenses_index_df['expenses_index'] = (expenses_index_df['amount'] / expenses_index_df['amount'].mean() * 100).round(2)
expenses_index_df['transaction_date'] = expenses_index_df['transaction_date'].dt.to_timestamp()
expenses_index_df['transaction_date'].dtype

### График 1 — динамика расходов по месяцам (индексированная)

In [ ]:
dynamics = sns.lineplot(
    expenses_index_df,
    x='transaction_date',
    y='expenses_index'
)
dynamics

### График 2 — сопоставление динамики расходов с инфляцией и ключевой ставкой


In [ ]:
merge_df['expenses_index'] = (merge_df['amount'] / merge_df['amount'].mean() * 100).round(2)
merge_df['transaction_date'] = merge_df['transaction_date'].dt.to_timestamp()
merge_df
expenses_index = sns.lineplot(merge_df, x='transaction_date', y='expenses_index')
expenses_index

In [ ]:
inflation_comparison = sns.lineplot(merge_df, x='transaction_date', y='inflation_rate')
inflation_comparison

In [ ]:
key_rate_comparison = sns.lineplot(merge_df, x='transaction_date', y='key_rate')
key_rate_comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 7))

sns.lineplot(data=merge_df, x='transaction_date', y='expenses_index', marker='o', ax=ax1, label='Индекс расходов')
ax1.axhline(100, color='gray', linestyle='--', linewidth=1)
ax1.set_ylabel('Индекс (среднее = 100)')

sns.lineplot(data=merge_df, x='transaction_date', y='inflation_rate', marker='o', ax=ax2, label='Инфляция')
sns.lineplot(data=merge_df, x='transaction_date', y='key_rate', marker='o', ax=ax2, label='Ключевая ставка')
ax2.set_ylabel('%')
ax2.set_xlabel('Месяц')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()